Install and load all dependencies (first time only) \
NOTE: you may need to restart the runtime afterwards (CTRL+M .).

In [1]:
!apt-get install -y \
    libgl1-mesa-dev \
    libgl1-mesa-glx \
    libglew-dev \
    libosmesa6-dev \
    software-properties-common

!apt-get install -y patchelf

!pip install gym
!pip install free-mujoco-py

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
software-properties-common is already the newest version (0.99.22.9).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
The following additional packages will be installed:
  libegl-dev libgl-dev libgles-dev libgles1 libglu1-mesa libglu1-mesa-dev
  libglvnd-core-dev libglvnd-dev libglx-dev libopengl-dev libosmesa6
The following NEW packages will be installed:
  libegl-dev libgl-dev libgl1-mesa-dev libgles-dev libgles1 libglew-dev
  libglu1-mesa libglu1-mesa-dev libglvnd-core-dev libglvnd-dev libglx-dev
  libopengl-dev libosmesa6 libosmesa6-dev
0 upgraded, 14 newly installed, 0 to remove and 34 not upgraded.
Need to get 4,008 kB of archives.
After this operation, 19.3 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libglx-dev amd64 1.4.0-1 [14.1 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/main amd64 libgl-dev amd64

Set up the custom Hopper environment



1.   Upload `classes.zip` to the current session's file storage
2.   Un-zip it by running cell below


In [5]:
!unzip classes.zip

unzip:  cannot find or open classes.zip, classes.zip.zip or classes.zip.ZIP.




---



\

**Train an RL agent on the OpenAI Gym Hopper environment using REINFORCE and Actor-critic algorithms**

\


TASK 2 and 3: interleave data collection to policy updates

In [6]:
import argparse

import torch
import numpy as np
import tensorflow as tf
import gym

from env.custom_hopper import *
from agent import Agent, Policy

ModuleNotFoundError: No module named 'env'

In [ ]:
n_episodes = 100000
print_every = 20000
device = 'cpu'

In [ ]:
env = gym.make('CustomHopper-source-v0')
# env = gym.make('CustomHopper-target-v0')

print('Action space:', env.action_space)
print('State space:', env.observation_space)
print('Dynamics parameters:', env.get_parameters())

Define the actor and critic networks

In [ ]:
actor = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(env.action_space.shape[0], activation='softmax')
	])

critic = tf.keras.Sequential([
    tf.keras.layers.Dense(32, activation='relu'),
    tf.keras.layers.Dense(1)
	])

Define optimizer and loss functions

In [ ]:
actor_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
critic_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)


In [ ]:
"""
  Training
"""
observation_space_dim = env.observation_space.shape[-1]
action_space_dim = env.action_space.shape[-1]

policy = Policy(observation_space_dim, action_space_dim)
agent = Agent(policy, device=device)

for episode in range(n_episodes):
  done = False
  episode_reward = 0
	state = env.reset()  # Reset the environment and observe the initial state

	with tf.GradientTape(persistent=True) as tape:
				for t in range(1, 10000):  # Limit the number of time steps
				# Choose an action using the actor
					action_probs = actor(np.array([state]))
					action = np.random.choice(env.action_space.shape[0], p=action_probs.numpy()[0])

					# Take the chosen action and observe the next state and reward
					next_state, reward, done, _ = env.step(action)

					# Compute the advantage
					state_value = critic(np.array([state]))[0, 0]
					next_state_value = critic(np.array([next_state]))[0, 0]
					advantage = reward + next_state_value - state_value

					# Compute actor and critic losses
					actor_loss = -tf.math.log(action_probs[0, action]) * advantage
					critic_loss = tf.square(advantage)

					episode_reward += reward

					# Update actor and critic
					actor_gradients = tape.gradient(actor_loss, actor.trainable_variables)
					critic_gradients = tape.gradient(critic_loss, critic.trainable_variables)
					actor_optimizer.apply_gradients(zip(actor_gradients, actor.trainable_variables))
					critic_optimizer.apply_gradients(zip(critic_gradients, critic.trainable_variables))

					if done:
                				break

  if (episode+1)%print_every == 0:
    print('Training episode:', episode)
    print('Episode return:', train_reward)



torch.save(agent.policy.state_dict(), "model.mdl")